# 📝 Phase 2: OCR + NLP Text Classification

**Samsung PRISM - Harmful Content Detection Pipeline**

This notebook implements text detection and context-aware classification:
- **EasyOCR** for text extraction
- **DistilBERT** for context classification

## Classification Labels
- 🟢 **SAFE** - Normal, harmless text
- 🟡 **PROMOTIONAL** - Sales, discounts, marketing text
- 🔴 **ABUSIVE** - Hate speech, threats, harmful language

---

**⚠️ IMPORTANT: Enable GPU in Colab**
- Go to `Runtime` → `Change runtime type` → Select `GPU`

## 1️⃣ Setup & Installation

In [ ]:
# Install required packages
!pip install easyocr>=1.7.1 -q
!pip install transformers>=4.35.0 -q
!pip install datasets -q
!pip install accelerate -q
!pip install scikit-learn -q

# Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Import libraries
import os
import json
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import easyocr
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

## 2️⃣ EasyOCR Setup

In [ ]:
# Initialize EasyOCR
print("Initializing EasyOCR (this may take a moment to download models)...")

# Create reader with English language support
ocr_reader = easyocr.Reader(
    ['en'],           # Languages
    gpu=True,         # Use GPU
    model_storage_directory='/content/ocr_models'
)

print("✓ EasyOCR initialized successfully")

In [ ]:
# OCR extraction function
def extract_text_from_image(image_path, min_confidence=0.3):
    """
    Extract text from image using EasyOCR.
    
    Args:
        image_path: Path to image file
        min_confidence: Minimum confidence threshold
        
    Returns:
        Dictionary with extracted text and details
    """
    # Read image
    if isinstance(image_path, str):
        image = cv2.imread(image_path)
    else:
        image = image_path
    
    # Run OCR
    results = ocr_reader.readtext(image)
    
    extracted_texts = []
    for (bbox, text, confidence) in results:
        if confidence >= min_confidence:
            # Convert bbox format
            x_coords = [p[0] for p in bbox]
            y_coords = [p[1] for p in bbox]
            
            extracted_texts.append({
                'text': text,
                'confidence': round(float(confidence), 4),
                'bbox': {
                    'x1': int(min(x_coords)),
                    'y1': int(min(y_coords)),
                    'x2': int(max(x_coords)),
                    'y2': int(max(y_coords))
                }
            })
    
    # Combine all text
    combined_text = ' '.join([t['text'] for t in extracted_texts])
    
    return {
        'text_found': len(extracted_texts) > 0,
        'combined_text': combined_text,
        'text_regions': extracted_texts,
        'total_regions': len(extracted_texts)
    }

print("✓ OCR extraction function defined")

In [ ]:
# Test OCR on a sample image
# Create a test image with text
from PIL import Image, ImageDraw, ImageFont

# Create sample promotional image
img = Image.new('RGB', (600, 400), color='white')
draw = ImageDraw.Draw(img)

# Add promotional text
try:
    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 40)
    font_small = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 24)
except:
    font = ImageFont.load_default()
    font_small = font

draw.text((50, 50), "BUY NOW!", fill='red', font=font)
draw.text((50, 120), "50% OFF - Limited Time!", fill='blue', font=font_small)
draw.text((50, 180), "Sale ends today!", fill='green', font=font_small)
draw.text((50, 250), "Free shipping on all orders", fill='black', font=font_small)

# Save test image
test_image_path = '/content/test_promotional.png'
img.save(test_image_path)

# Run OCR
ocr_result = extract_text_from_image(test_image_path)

print("\n📋 OCR Results:")
print(f"Text Found: {ocr_result['text_found']}")
print(f"Combined Text: {ocr_result['combined_text']}")
print(f"Total Regions: {ocr_result['total_regions']}")

# Display image
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.title("Test Image for OCR")
plt.axis('off')
plt.show()

## 3️⃣ NLP Text Classification - Training Data Preparation

In [ ]:
# Create training dataset for text classification
# Labels: 0=SAFE, 1=PROMOTIONAL, 2=ABUSIVE

training_data = {
    'text': [],
    'label': []
}

# SAFE examples (label=0)
safe_texts = [
    "Welcome to our store",
    "Thank you for visiting",
    "Product information and specifications",
    "Contact us for more details",
    "Open Monday to Friday",
    "Read our privacy policy",
    "Terms and conditions apply",
    "Please read the instructions carefully",
    "For customer support call this number",
    "Made with high quality materials",
    "This product comes with warranty",
    "User manual included in the package",
    "Assembled in the USA",
    "Check our website for more info",
    "Follow us on social media",
    "New arrivals available",
    "Browse our collection",
    "Quality guaranteed",
    "Customer satisfaction is our priority",
    "Easy returns within 30 days",
]

# PROMOTIONAL examples (label=1)
promotional_texts = [
    "BUY NOW - 50% OFF!",
    "Limited time offer - Act fast!",
    "Sale ends today!",
    "Free shipping on all orders",
    "Discount code: SAVE20",
    "Flash sale - Everything must go!",
    "Best price guaranteed",
    "Order now and save big",
    "Exclusive deal for new customers",
    "Hurry! Only 5 left in stock",
    "Black Friday deals - Up to 70% off",
    "Buy one get one free",
    "Special offer ends midnight",
    "Clearance sale - Final markdown",
    "Members get extra 10% off",
    "Use promo code for discount",
    "Lowest prices of the season",
    "Mega sale event now live",
    "Save up to $500 today only",
    "Don't miss this incredible deal",
    "Price drop alert!",
    "Biggest sale of the year",
    "Bundle and save more",
    "Limited stock at these prices",
    "Early bird special",
]

# ABUSIVE examples (label=2)
abusive_texts = [
    "You're so stupid and worthless",
    "I hate everyone here",
    "This is complete garbage",
    "You're an absolute idiot",
    "Shut up and go away",
    "You're pathetic and useless",
    "Nobody likes you here",
    "You should be ashamed of yourself",
    "What a waste of space",
    "You're a complete failure",
    "Go away, nobody wants you",
    "You're disgusting",
    "Worst thing I've ever seen",
    "You're a total loser",
    "This is trash and so are you",
]

# Add to training data
for text in safe_texts:
    training_data['text'].append(text)
    training_data['label'].append(0)

for text in promotional_texts:
    training_data['text'].append(text)
    training_data['label'].append(1)

for text in abusive_texts:
    training_data['text'].append(text)
    training_data['label'].append(2)

# Create DataFrame
df = pd.DataFrame(training_data)

# Shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Training Data Statistics:")
print(f"Total samples: {len(df)}")
print(f"\nClass distribution:")
label_names = {0: 'SAFE', 1: 'PROMOTIONAL', 2: 'ABUSIVE'}
for label, count in df['label'].value_counts().sort_index().items():
    print(f"  {label_names[label]}: {count}")

df.head(10)

In [ ]:
# Split into train/validation
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")

## 4️⃣ DistilBERT Model Training

In [ ]:
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
import torch

# Load tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

print("✓ DistilBERT tokenizer loaded")

In [ ]:
# Create datasets
train_dataset = Dataset.from_dict({
    'text': train_texts,
    'label': train_labels
})

val_dataset = Dataset.from_dict({
    'text': val_texts,
    'label': val_labels
})

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

# Tokenize datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

print("✓ Datasets tokenized")

In [ ]:
# Load model
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=3,
    id2label={0: 'SAFE', 1: 'PROMOTIONAL', 2: 'ABUSIVE'},
    label2id={'SAFE': 0, 'PROMOTIONAL': 1, 'ABUSIVE': 2}
)

print("✓ DistilBERT model loaded")
print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
# Training configuration
training_args = TrainingArguments(
    output_dir='/content/text_classifier',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir='/content/logs',
    logging_steps=10,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    learning_rate=2e-5,
    fp16=True,  # Mixed precision for faster training
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")

In [ ]:
# Metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [ ]:
# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("✓ Trainer initialized")

In [ ]:
# 🚀 Start training
print("\n" + "="*50)
print("🚀 STARTING TEXT CLASSIFIER TRAINING")
print("="*50 + "\n")

trainer.train()

print("\n" + "="*50)
print("✓ TRAINING COMPLETED")
print("="*50)

## 5️⃣ Model Evaluation

In [ ]:
# Evaluate on validation set
eval_results = trainer.evaluate()

print("\n" + "="*50)
print("📊 EVALUATION RESULTS")
print("="*50)
print(f"Loss:      {eval_results['eval_loss']:.4f}")
print(f"Accuracy:  {eval_results['eval_accuracy']:.4f}")
print(f"Precision: {eval_results['eval_precision']:.4f}")
print(f"Recall:    {eval_results['eval_recall']:.4f}")
print(f"F1 Score:  {eval_results['eval_f1']:.4f}")
print("="*50)

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Get predictions
predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = val_labels

# Confusion matrix
cm = confusion_matrix(labels, preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['SAFE', 'PROMOTIONAL', 'ABUSIVE'],
            yticklabels=['SAFE', 'PROMOTIONAL', 'ABUSIVE'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Text Classification Confusion Matrix')
plt.savefig('/content/confusion_matrix_text.png', dpi=150)
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(labels, preds, target_names=['SAFE', 'PROMOTIONAL', 'ABUSIVE']))

## 6️⃣ Inference Pipeline

In [ ]:
# Text classification function
def classify_text(text, model, tokenizer, device='cuda'):
    """
    Classify text using trained DistilBERT model.
    
    Args:
        text: Text to classify
        model: Trained model
        tokenizer: Tokenizer
        device: 'cuda' or 'cpu'
        
    Returns:
        Classification result with label and confidence
    """
    label_map = {0: 'SAFE', 1: 'PROMOTIONAL', 2: 'ABUSIVE'}
    
    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=128,
        padding=True
    )
    
    # Move to device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model = model.to(device)
    model.eval()
    
    # Inference
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        pred_class = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][pred_class].item()
    
    return {
        'label': label_map[pred_class],
        'label_id': pred_class,
        'confidence': round(confidence, 4),
        'all_scores': {
            label_map[i]: round(probs[0][i].item(), 4)
            for i in range(3)
        }
    }

print("✓ Classification function defined")

In [ ]:
# Full OCR + NLP pipeline
def detect_text_in_image(image_path, model, tokenizer):
    """
    Full pipeline: OCR extraction + NLP classification.
    
    Args:
        image_path: Path to image
        model: Trained NLP model
        tokenizer: Tokenizer
        
    Returns:
        Complete detection and classification result
    """
    # Step 1: Extract text with OCR
    ocr_result = extract_text_from_image(image_path)
    
    if not ocr_result['text_found']:
        return {
            'text_found': False,
            'combined_text': '',
            'classification': {
                'label': 'SAFE',
                'confidence': 1.0,
                'reason': 'No text detected in image'
            }
        }
    
    # Step 2: Classify extracted text
    combined_text = ocr_result['combined_text']
    classification = classify_text(combined_text, model, tokenizer)
    
    return {
        'text_found': True,
        'combined_text': combined_text,
        'text_regions': ocr_result['text_regions'],
        'classification': classification,
        'decision': 'UNSAFE' if classification['label'] != 'SAFE' else 'SAFE'
    }

print("✓ Full pipeline function defined")

In [ ]:
# Test the full pipeline
print("\n" + "="*50)
print("🧪 TESTING FULL OCR + NLP PIPELINE")
print("="*50)

# Test on the promotional image we created earlier
result = detect_text_in_image(test_image_path, model, tokenizer)

print(f"\n📋 Results for: {test_image_path}")
print(f"Text Found: {result['text_found']}")
print(f"Extracted Text: {result['combined_text']}")
print(f"\nClassification:")
print(f"  Label: {result['classification']['label']}")
print(f"  Confidence: {result['classification']['confidence']:.2%}")
print(f"\n🚨 FINAL DECISION: {result['decision']}")

In [ ]:
# Test with more examples
test_texts = [
    "Welcome to our website",
    "BUY NOW - 80% OFF - LIMITED TIME!",
    "You're an idiot and worthless",
    "Product specifications and details",
    "Flash sale - Order now and save!"
]

print("\n" + "="*50)
print("🧪 TESTING TEXT CLASSIFICATION")
print("="*50 + "\n")

for text in test_texts:
    result = classify_text(text, model, tokenizer)
    emoji = {'SAFE': '🟢', 'PROMOTIONAL': '🟡', 'ABUSIVE': '🔴'}[result['label']]
    print(f"{emoji} [{result['label']}] ({result['confidence']:.2%}): \"{text}\"")

## 7️⃣ Save Model

In [ ]:
# Save model to Google Drive
drive_path = '/content/drive/MyDrive/samsung_prism/models/text_classifier'
os.makedirs(drive_path, exist_ok=True)

# Save model and tokenizer
model.save_pretrained(drive_path)
tokenizer.save_pretrained(drive_path)

print(f"\n✓ Model saved to: {drive_path}")
print("\nSaved files:")
for f in os.listdir(drive_path):
    size_kb = os.path.getsize(f'{drive_path}/{f}') / 1024
    print(f"  - {f} ({size_kb:.2f} KB)")

In [ ]:
# Example output format
sample_output = {
    "status": "UNSAFE",
    "detection_type": "text",
    "text_found": True,
    "extracted_text": "BUY NOW - 50% OFF!",
    "classification": {
        "label": "PROMOTIONAL",
        "confidence": 0.94,
        "all_scores": {
            "SAFE": 0.02,
            "PROMOTIONAL": 0.94,
            "ABUSIVE": 0.04
        }
    },
    "explanation": "Promotional content detected with 94% confidence"
}

print("\n📋 Sample Output Format:")
print(json.dumps(sample_output, indent=2))

In [ ]:
print("\n✓ OCR + NLP Text Classification Complete!")
print("\nNext Steps:")
print("  1. Run 03_logo_detection.ipynb")
print("  2. Run 04_unified_pipeline.ipynb for end-to-end inference")

---

## Summary

| Component | Details |
|-----------|--------|
| OCR Engine | EasyOCR |
| NLP Model | DistilBERT |
| Classes | SAFE, PROMOTIONAL, ABUSIVE |
| Accuracy | [See evaluation above] |

**Model saved to:** `/content/drive/MyDrive/samsung_prism/models/text_classifier/`